# Health Data Extraction

From Garmin watch

In [1]:
import os
import sys
import importlib
import datetime
import requests
import getpass
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np

from garth.exc import GarthException, GarthHTTPError
from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

import lib

importlib.reload(lib)

project = lib.Project()
log = lib.getLogger(project.name)

DATE_FORMAT = '%Y-%m-%d'


# Garmin helper functions to interact with API
Taken from example file provided in documentation to ensure safe usage with API

In [2]:
def safe_api_call(api_method, *args, **kwargs):
    """
    Safe API call wrapper with comprehensive error handling.

    This demonstrates the error handling patterns used throughout the library.
    Returns (success: bool, result: Any, error_message: str)
    """
    try:
        result = api_method(*args, **kwargs)
        return True, result, None

    except GarthHTTPError as e:
        # Handle specific HTTP errors gracefully
        error_str = str(e)
        status_code = getattr(getattr(e, "response", None), "status_code", None)

        if status_code == 400 or "400" in error_str:
            return (
                False,
                None,
                "Endpoint not available (400 Bad Request) - Feature may not be enabled for your account",
            )
        elif status_code == 401 or "401" in error_str:
            return (
                False,
                None,
                "Authentication required (401 Unauthorized) - Please re-authenticate",
            )
        elif status_code == 403 or "403" in error_str:
            return (
                False,
                None,
                "Access denied (403 Forbidden) - Account may not have permission",
            )
        elif status_code == 404 or "404" in error_str:
            return (
                False,
                None,
                "Endpoint not found (404) - Feature may have been moved or removed",
            )
        elif status_code == 429 or "429" in error_str:
            return (
                False,
                None,
                "Rate limit exceeded (429) - Please wait before making more requests",
            )
        elif status_code == 500 or "500" in error_str:
            return (
                False,
                None,
                "Server error (500) - Garmin's servers are experiencing issues",
            )
        elif status_code == 503 or "503" in error_str:
            return (
                False,
                None,
                "Service unavailable (503) - Garmin's servers are temporarily unavailable",
            )
        else:
            return False, None, f"HTTP error: {e}"

    except FileNotFoundError:
        return (
            False,
            None,
            "No valid tokens found. Please login with your email/password to create new tokens.",
        )

    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication issue: {e}"

    except GarminConnectConnectionError as e:
        return False, None, f"Connection issue: {e}"

    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit exceeded: {e}"

    except Exception as e:
        return False, None, f"Unexpected error: {e}"

def get_credentials():
    """Get email and password from environment or user input."""
    email = os.getenv("EMAIL")
    password = os.getenv("PASSWORD")

    if not email:
        email = input("Login email: ")
    if not password:
        password = getpass("Enter password: ")

    return email, password

def init_api() -> Garmin | None:
    """Initialize Garmin API with authentication and token management."""

    # Configure token storage
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = Path(tokenstore).expanduser()

    print(f"🔐 Token storage: {tokenstore_path}")

    # Check if token files exist
    if tokenstore_path.exists():
        print("📄 Found existing token directory")
        token_files = list(tokenstore_path.glob("*.json"))
        if token_files:
            print(
                f"🔑 Found {len(token_files)} token file(s): {[f.name for f in token_files]}"
            )
        else:
            print("⚠️ Token directory exists but no token files found")
    else:
        print("📭 No existing token directory found")

    # First try to login with stored tokens
    try:
        print("🔄 Attempting to use saved authentication tokens...")
        garmin = Garmin()
        garmin.login(str(tokenstore_path))
        print("✅ Successfully logged in using saved tokens!")
        return garmin

    except (
        FileNotFoundError,
        GarthHTTPError,
        GarminConnectAuthenticationError,
        GarminConnectConnectionError,
    ):
        print("🔑 No valid tokens found. Requesting fresh login credentials.")

    # Loop for credential entry with retry on auth failure
    while True:
        try:
            # Get credentials
            email, password = get_credentials()

            print("� Logging in with credentials...")
            garmin = Garmin(
                email=email, password=password, is_cn=False, return_on_mfa=True
            )
            result1, result2 = garmin.login()

            if result1 == "needs_mfa":
                print("🔐 Multi-factor authentication required")

                mfa_code = input("Please enter your MFA code: ")
                print("🔄 Submitting MFA code...")

                try:
                    garmin.resume_login(result2, mfa_code)
                    print("✅ MFA authentication successful!")

                except GarthHTTPError as garth_error:
                    # Handle specific HTTP errors from MFA
                    error_str = str(garth_error)
                    if "429" in error_str and "Too Many Requests" in error_str:
                        print("❌ Too many MFA attempts")
                        print("💡 Please wait 30 minutes before trying again")
                        sys.exit(1)
                    elif "401" in error_str or "403" in error_str:
                        print("❌ Invalid MFA code")
                        print("💡 Please verify your MFA code and try again")
                        continue
                    else:
                        # Other HTTP errors - don't retry
                        print(f"❌ MFA authentication failed: {garth_error}")
                        sys.exit(1)

                except GarthException as garth_error:
                    print(f"❌ MFA authentication failed: {garth_error}")
                    print("💡 Please verify your MFA code and try again")
                    continue

            # Save tokens for future use
            garmin.garth.dump(str(tokenstore_path))
            print(f"💾 Authentication tokens saved to: {tokenstore_path}")
            print("✅ Login successful!")
            return garmin

        except GarminConnectAuthenticationError:
            print("❌ Authentication failed:")
            print("💡 Please check your username and password and try again")
            # Continue the loop to retry
            continue

        except (
            FileNotFoundError,
            GarthHTTPError,
            GarminConnectConnectionError,
            requests.exceptions.HTTPError,
        ) as err:
            print(f"❌ Connection error: {err}")
            print("💡 Please check your internet connection and try again")
            return None

        except KeyboardInterrupt:
            print("\n👋 Cancelled by user")
            return None


# Get Garmin client

In [3]:
api = init_api()

🔐 Token storage: /Users/christophermagno/.garminconnect
📄 Found existing token directory
🔑 Found 2 token file(s): ['oauth2_token.json', 'oauth1_token.json']
🔄 Attempting to use saved authentication tokens...
✅ Successfully logged in using saved tokens!


# Helper functions for datetime

In [4]:
def _get_date_string(date):
    return date.strftime(DATE_FORMAT)

def today():
    return datetime.date.today().strftime(DATE_FORMAT)


def get_date_range(start=None, rng=None):
    start = start or datetime.datetime.today()
    rng = rng or int(start.strftime('%j'))
    dates = reversed([start - datetime.timedelta(days=x) for x in range(rng)])
    return [x.strftime(DATE_FORMAT) for x in dates]


)# Helper functions to gather ando organize Garmin data

Some useful methods from the Garmin class to use
* get_stats - using
* get_steps_data
* get_daily_steps
* get_floors
* get_heart_rates - using
* get_sleep_data - using
* get_stress_data
* get_rhr_day
* get_hrv_data
* get_fitnessage_data - using

Others toook at
* get_activities
* get_activities_fordate
* get_earned_badges

In [123]:
def get_sleep_data(date):

    sleep_data = {}

    to_pop = [
        'id',
        'userProfilePK',
        'napTimeSeconds',
        'sleepWindowConfirmed',
        'sleepWindowConfirmationType',
        'autoSleepStartTimestampGMT',
        'autoSleepEndTimestampGMT',
        'sleepQualityTypePK',
        'sleepResultTypePK',
        'deviceRemCapable',
        'retro',
        'sleepFromDevice',
        'sleepScores',
        'sleepScoreInsight',
        'sleepScorePersonalizedInsight',
        'sleepVersion'
    ]

    data = safe_api_call(api.get_sleep_data, date)[1]

    sleep_data.update(data['dailySleepDTO'])
    if 'sleepScores' in sleep_data:
        sleep_data['sleepScore'] = sleep_data['sleepScores']['overall']['value']
        sleep_data['sleepScoreQuality'] = sleep_data['sleepScores']['overall']['qualifierKey']
        sleep_data['stressSleepQuality'] = sleep_data['sleepScores']['stress']['qualifierKey']
        sleep_data['awakeCountQuality'] = sleep_data['sleepScores']['awakeCount']['qualifierKey']
        sleep_data['remSleepQuality'] = sleep_data['sleepScores']['remPercentage']['qualifierKey']
        sleep_data['restlessnessSleepQuality'] = sleep_data['sleepScores']['restlessness']['qualifierKey']
        sleep_data['lightSleepQuality'] = sleep_data['sleepScores']['lightPercentage']['qualifierKey']
        sleep_data['deepSleepQuality'] = sleep_data['sleepScores']['deepPercentage']['qualifierKey']
    else:
        sleep_data['sleepScore'] = None
        sleep_data['sleepScoreQuality'] = None
        sleep_data['stressSleepQuality'] = None
        sleep_data['awakeCountQuality'] = None
        sleep_data['remSleepQuality'] = None
        sleep_data['restlessnessSleepQuality'] = None
        sleep_data['lightSleepQuality'] = None
        sleep_data['deepSleepQuality'] = None

    # result['sleepHeartRate'] = data['sleepHeartRate']
    sleep_data['avgOvernightHrv'] = data.get('avgOvernightHrv')
    sleep_data['hrvStatus'] = data.get('hrvStatus')
    sleep_data['restingHeartRate'] = data.get('restingHeartRate')

    for key in to_pop:
        try:
            sleep_data.pop(key)
        except KeyError as e:
            pass

    return sleep_data

def get_hydration_data(date):
    data = safe_api_call(api.get_hydration_data, date)[1]
    hydration_data = {
        'hydrationValueInML': data['valueInML'],
        'hydrationGoalInML': data['goalInML'],
        'sweatLossInML': data['sweatLossInML']
    }
    return hydration_data


In [121]:
def get_health_data(date):
    """
    """

    to_pop = [
        'userProfileId',
        'userDailySummaryId',
        'burnedKilocalories',
        'wellnessActiveKilocalories',
        'netRemainingKilocalories',
        'rule',
        'wellnessStartTimeGmt',
        'wellnessStartTimeLocal',
        'wellnessEndTimeGmt',
        'wellnessEndTimeLocal',
        'durationInMilliseconds',
        'wellnessDescription',
        'includesWellnessData',
        'includesActivityData',
        'includesCalorieConsumedData',
        'privacyProtected',
        'floorsAscended',
        'floorsDescended',
        'lastSevenDaysAvgRestingHeartRate',
        'source',
        'lastSyncTimestampGMT',
        'bodyBatteryMostRecentValue',
        'bodyBatteryVersion',
        'averageSpo2',
        'lowestSpo2',
        'latestSpo2',
        'latestSpo2ReadingTimeGmt',
        'latestSpo2ReadingTimeLocal',
        'latestSpo2ReadingTimeLocalaverageMonitoringEnvironmentAltitude',
        'restingCaloriesFromActivity',
        'latestRespirationValue',
        'latestRespirationTimeGMT',
        'respirationAlgorithmVersion',
        'ageGroup',
        'averageMonitoringEnvironmentAltitude',
        'bodyBatteryChargedValue',
        'bodyBatteryDrainedValue',
        'bodyBatteryHighestValue',
        'bodyBatteryLowestValue',
        'bodyBatteryDuringSleep',
        'wellnessKilocalories',
        'consumedKilocalories',
        'remainingKilocalories',
        'netCalorieGoal',
        'wellnessDistanceMeters',
        'userNote',
        'sleepingSeconds',
        'minAvgHeartRate',
        'maxAvgHeartRate',
        'abnormalHeartRateAlertsCount',
        'unmeasurableSleepSeconds',
        'measurableAsleepDuration',
        'measurableAwakeDuration',
        'stressPercentage',
        'restStressPercentage',
        'activityStressPercentage',
        'uncategorizedStressPercentage',
        'lowStressPercentage',
        'mediumStressPercentage',
        'highStressPercentage',
        'restStressDuration',
        'userFloorsAscendedGoal',
    ]

    health_data = safe_api_call(api.get_stats, date)[1]
    health_data['fitnessAge'] = int(safe_api_call(api.get_fitnessage_data, date)[1]['fitnessAge'])

    # Heart rate data
    hdata = safe_api_call(api.get_heart_rates, date)[1]['heartRateValues']
    if hdata:
        heart_rates = [v[1] for v in hdata if v[1] is not None]
        health_data['avgHeartRate'] = float(np.array(heart_rates).mean())

    # Get sleep data
    health_data.update(get_sleep_data(date))

    # Get hydration data
    health_data.update(get_hydration_data(date))

    for key in to_pop:
        try:
            health_data.pop(key)
        except KeyError as e:
            pass

    # Convert/Add some columns
    convert_dict = {}
    for key, value in health_data.items():
        if value:
            if 'Meters' in key:
                convert_dict[key.replace('Meters', 'Miles')] = value / 1609
            elif 'Seconds' in key:
                convert_dict[key.replace('Seconds', 'Hours')] = value / 3600
            elif 'Duration' in key:
                convert_dict[key.replace('Duration', 'Hours')] = value / 3600
            elif 'Minutes' in key:
                convert_dict[key.replace('Minutes', 'Hours')] = value / 60
            elif 'InML' in key:
                convert_dict[key.replace('InML', 'InCups')] = value / 240

    # Update
    health_data.update(convert_dict)

    # Set columns to title format
    health_data_clean = {}
    for key, value in health_data.items():
        if isinstance(value, str):
            value = value.title().replace('_', ' ')
        health_data_clean[lib.convert_camel_case(key).title()] = value

    return health_data_clean.pop('Uuid'), health_data_clean

# Get Health Data

In [122]:
dates = get_date_range()

In [124]:
overall_data = {}
for date in tqdm(dates):
    id, health_data = get_health_data(date)
    overall_data[id] = health_data


100%|██████████| 360/360 [03:04<00:00,  1.95it/s]


# Create the Health Dataframe

In [127]:
df = pd.DataFrame(overall_data).T.convert_dtypes()

In [128]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 360 entries, 8F2Ddf09F3E04Ac780A5470C4099A803 to 42887A17D6674Cf4Ab5Cfd734D5Ff65E
Data columns (total 84 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Total Kilocalories             360 non-null    Int64  
 1   Active Kilocalories            360 non-null    Int64  
 2   Bmr Kilocalories               360 non-null    Int64  
 3   Total Steps                    358 non-null    Int64  
 4   Total Distance Meters          358 non-null    Int64  
 5   Calendar Date                  360 non-null    string 
 6   Daily Step Goal                360 non-null    Int64  
 7   Highly Active Seconds          360 non-null    Int64  
 8   Active Seconds                 360 non-null    Int64  
 9   Sedentary Seconds              360 non-null    Int64  
 10  Moderate Intensity Minutes     360 non-null    Int64  
 11  Vigorous Intensity Minutes     360 non-null    Int64  


In [129]:
df.columns.tolist()

['Total Kilocalories',
 'Active Kilocalories',
 'Bmr Kilocalories',
 'Total Steps',
 'Total Distance Meters',
 'Calendar Date',
 'Daily Step Goal',
 'Highly Active Seconds',
 'Active Seconds',
 'Sedentary Seconds',
 'Moderate Intensity Minutes',
 'Vigorous Intensity Minutes',
 'Floors Ascended In Meters',
 'Floors Descended In Meters',
 'Intensity Minutes Goal',
 'Min Heart Rate',
 'Max Heart Rate',
 'Resting Heart Rate',
 'Average Stress Level',
 'Max Stress Level',
 'Stress Duration',
 'Activity Stress Duration',
 'Uncategorized Stress Duration',
 'Total Stress Duration',
 'Low Stress Duration',
 'Medium Stress Duration',
 'High Stress Duration',
 'Stress Qualifier',
 'Body Battery At Wake Time',
 'Avg Waking Respiration Value',
 'Highest Respiration Value',
 'Lowest Respiration Value',
 'Fitness Age',
 'Sleep Time Seconds',
 'Sleep Start Timestamp G M T',
 'Sleep End Timestamp G M T',
 'Sleep Start Timestamp Local',
 'Sleep End Timestamp Local',
 'Deep Sleep Seconds',
 'Light Sleep 

# Convert some types

In [143]:
df['Date'] = pd.to_datetime(df['Date'])
for col in ['Sleep Start Timestamp GMT', 'Sleep End Timestamp GMT', 'Sleep Start Timestamp Local', 'Sleep End Timestamp Local']:
    df[col] = pd.to_datetime(df[col], unit='ms')

In [146]:
remap_columns = {
    'Calendar Date': 'Date',

    'Fitness Age': 'Fitness Age',

    # Calories
    'Total Kilocalories': 'Calories',
    'Active Kilocalories': 'Active Calories',
    'Bmr Kilocalories': 'Resting Calories',

    # Hydration
    'Hydration Value In M L': 'Hydration Value In ML',
    'Hydration Goal In M L': 'Hydration Goal In ML',
    'Sweat Loss In M L': 'Sweat Loss In ML',
    'Hydration Value In Cups': 'Hydration Value In Cups',
    'Hydration Goal In Cups': 'Hydration Goal In Cups',
    'Sweat Loss In Cups': 'Sweat Loss In Cups',

    # Heart Rate
    'Avg Heart Rate': 'Average Heart Rate',
    'Min Heart Rate': 'Min Heart Rate',
    'Max Heart Rate': 'Max Heart Rate',
    'Resting Heart Rate': 'Resting Heart Rate',
    'Hrv Status': 'Heart Rate Variability Qualifier',

    # Respiration
    'Avg Waking Respiration Value': 'Avg Waking Respiration Value',
    'Highest Respiration Value': 'Highest Respiration Value',
    'Lowest Respiration Value': 'Lowest Respiration Value',

    # Steps/Distance
    'Total Steps': 'Total Steps',
    'Total Distance Meters': 'Total Distance Meters',
    'Total Distance Miles': 'Total Distance Miles',
    'Daily Step Goal': 'Daily Step Goal',

    # Floors
    'Floors Ascended In Meters': 'Floors Ascended In Meters',
    'Floors Descended In Meters': 'Floors Descended In Meters',

    'Floors Ascended In Miles': 'Floors Ascended In Miles',
    'Floors Descended In Miles': 'Floors Descended In Miles',

    # Activity
    'Active Seconds': 'Active Seconds',
    'Highly Active Seconds': 'Highly Active Seconds',
    'Sedentary Seconds': 'Sedentary Seconds',
    'Moderate Intensity Minutes': 'Moderate Intensity Minutes',
    'Vigorous Intensity Minutes': 'Vigorous Intensity Minutes',
    'Intensity Minutes Goal': 'Intensity Minutes Goal',

    'Active Hours': 'Active Hours',
    'Highly Active Hours': 'Highly Active Hours',
    'Sedentary Hours': 'Sedentary Hours',
    'Moderate Intensity Hours': 'Moderate Intensity Hours',
    'Vigorous Intensity Hours': 'Vigorous Intensity Hours',
    'Intensity Hours Goal': 'Intensity Hours Goal',

    # Stress
    'Average Stress Level': 'Average Stress Level',
    'Total Stress Duration': 'Total Stress Seconds',
    'Stress Duration': 'Stress Seconds',
    'Max Stress Level': 'Max Stress Level',
    'Uncategorized Stress Duration': 'Uncategorized Stress Seconds',
    'Low Stress Duration': 'Low Stress Seconds',
    'Medium Stress Duration': 'Medium Stress Seconds',
    'High Stress Duration': 'High Stress Seconds',
    'Activity Stress Duration': 'Activity Stress Seconds',
    'Stress Qualifier': 'Stress Qualifier',

    'Stress Hours': 'Stress Hours',
    'Activity Stress Hours': 'Activity Stress Hours',
    'Uncategorized Stress Hours': 'Uncategorized Stress Hours',
    'Total Stress Hours': 'Total Stress Hours',
    'Low Stress Hours': 'Low Stress Hours',
    'Medium Stress Hours': 'Medium Stress Hours',
    'High Stress Hours': 'High Stress Hours',

    # Body battery
    'Body Battery At Wake Time': 'Body Battery',

    # Sleep data
    'Sleep Start Timestamp G M T': 'Sleep Start Timestamp GMT',
    'Sleep End Timestamp G M T': 'Sleep End Timestamp GMT',
    'Sleep Start Timestamp Local': 'Sleep Start Timestamp Local',
    'Sleep End Timestamp Local': 'Sleep End Timestamp Local',

    'Sleep Time Seconds': 'Sleep Time Seconds',
    'Sleep Score': 'Sleep Score',
    'Sleep Score Quality': 'Sleep Quality',
    'Sleep Score Feedback': 'Sleep Feedback',

    'Light Sleep Seconds': 'Light Sleep Seconds',
    'Deep Sleep Seconds': 'Deep Sleep Seconds',
    'Rem Sleep Seconds': 'Rem Sleep Seconds',
    'Awake Sleep Seconds': 'Awake Sleep Seconds',

    'Sleep Time Hours': 'Sleep Time Hours',
    'Deep Sleep Hours': 'Deep Sleep Hours',
    'Light Sleep Hours': 'Light Sleep Hours',
    'Rem Sleep Hours': 'Rem Sleep Hours',
    'Awake Sleep Hours': 'Awake Sleep Hours',

    'Average Respiration Value': 'Average Respiration Value',
    'Awake Count': 'Awake Count',
    'Avg Sleep Stress': 'Avg Sleep Stress',

    'Stress Sleep Quality': 'Stress Sleep Quality',
    'Awake Count Quality': 'Awake Count Quality',
    'Rem Sleep Quality': 'Rem Sleep Quality',
    'Restlessness Sleep Quality': 'Restlessness Sleep Quality',
    'Light Sleep Quality': 'Light Sleep Quality',
    'Deep Sleep Quality': 'Deep Sleep Quality',

    'Avg Overnight Hrv': 'Average Overnight Hrv',
}

In [ ]:
df = df[remap_columns.keys()]
df = df.rename(columns=remap_columns)
df

In [145]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 360 entries, 8F2Ddf09F3E04Ac780A5470C4099A803 to 42887A17D6674Cf4Ab5Cfd734D5Ff65E
Data columns (total 85 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Date                              360 non-null    datetime64[ns]
 1   Fitness Age                       360 non-null    Int64         
 2   Calories                          360 non-null    Int64         
 3   Active Calories                   360 non-null    Int64         
 4   Resting Calories                  360 non-null    Int64         
 5   Hydration Value In ML             359 non-null    Float64       
 6   Hydration Goal In ML              360 non-null    Float64       
 7   Sweat Loss In ML                  52 non-null     Int64         
 8   Hydration Value In Cups           359 non-null    Float64       
 9   Hydration Goal In Cups            360 non-null    Float64       


In [149]:
lib.hasnull(df)

,Hydration Value In ML,Sweat Loss In ML,Hydration Value In Cups,Sweat Loss In Cups,Average Heart Rate,Resting Heart Rate,Heart Rate Variability Qualifier,Total Steps,Total Distance Meters,Total Distance Miles,...,Average Respiration Value,Awake Count,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv
8F2Ddf09F3E04Ac780A5470C4099A803,2880.0,98,12.0,0.408333,<NA>,50,<NA>,10941,8038,4.995649,...,13,2,11,Excellent,Fair,Excellent,Fair,Fair,Fair,<NA>
4Ab88Cfdf283481Fa593979341Aa8332,946.353,<NA>,3.943137,<NA>,<NA>,55,<NA>,7435,5578,3.46675,...,13,0,35,Poor,Excellent,Fair,Excellent,Excellent,Excellent,<NA>
A6433536E16A492E8A7655909869Ea9C,946.353,213,3.943137,0.8875,<NA>,52,<NA>,5793,4200,2.610317,...,13,3,14,Good,Fair,Excellent,Fair,Good,Fair,<NA>
8709A878C8B8459F8769C1Ff58D886D3,3785.41,<NA>,15.772542,<NA>,<NA>,52,<NA>,5188,3743,2.32629,...,13,2,17,Fair,Fair,Fair,Fair,Fair,Excellent,<NA>
E929E59Be1114B58A17D010D389233Ed,3785.41,<NA>,15.772542,<NA>,<NA>,57,<NA>,13070,9566,5.945308,...,14,4,41,Poor,Poor,Poor,Poor,Poor,Fair,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9Cffcfe0C75A4F229C72E6F21058999B,2880.0,<NA>,12.0,<NA>,74.820833,53,Balanced,6773,4920,3.0578,...,14,1,21,Fair,Excellent,Fair,Good,Excellent,Good,51
36715949D50C48398F9491E715Cba11C,2880.0,<NA>,12.0,<NA>,72.0,48,Balanced,7190,5119,3.181479,...,15,1,11,Excellent,Good,Fair,Good,Good,Excellent,59
C57645F966Ba4337Ade488C18A602F94,946.353,<NA>,3.943137,<NA>,69.657658,<NA>,<NA>,2818,2006,1.246737,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
B260B71813644B21Ad0950F3D652B7E0,2880.0,<NA>,12.0,<NA>,78.890278,55,Balanced,10406,7409,4.604723,...,14,5,35,Poor,Poor,Poor,Poor,Poor,Fair,45


# Export the data

In [148]:
df.to_csv(project.raw_file)